Parameter optimization
================

This notebook shows how both hyperparameters and data source weights can be optimized. This code merely serves as a demonstration of the process. For the full optimization code see [run optimization](https://github.com/saeyslab/nichenetpy/blob/main/optimization/run_optimization.py) and [combine optimization folds](https://github.com/saeyslab/nichenetpy/blob/main/optimization/combine_optimization_folds.ipynb). 

In [7]:
from nichenetpy.utils import read_csv_cols
from nichenetpy.parameter_optimization import construct_and_evaluate
from nichenetpy.evaluation import EvaluationData

from optuna import create_study
from optuna.trial import Trial
from optuna.samplers import GPSampler
from itertools import chain

import os
import requests
import pandas as pd
import session_info
import json
import numpy as np

In [8]:
network_path = os.path.normpath("./tutorial_files/model_construction/human")
if not os.path.exists(network_path):
    os.makedirs(network_path)
for filename in ("gr_human.csv", "lr_network_human.csv", "lr_sig_human.csv"):
    file_path = os.path.join(network_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)
train_path = os.path.normpath("./tutorial_files/model_optimization")
if not os.path.exists(train_path):
    os.makedirs(train_path)
for filename in (
    "settings_training_f1234.json",
    "settings_training_f1235.json",
    "settings_training_f1245.json",
    "settings_training_f1345.json",
    "settings_training_f2345.json",
):
    file_path = os.path.join(train_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/15799578/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)

In [9]:
gr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "gr_human.csv")))
lr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_network_human.csv")))
sig_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_sig_human.csv")))

In [10]:
with open(os.path.join(train_path, "settings_training_f1234.json"), "rb") as file:
    settings_CV = json.loads(file.read())
settings = settings_CV["settings"]
evaluation_data = EvaluationData(settings)

In [11]:
gr_network = gr_network[
    ~(
        (gr_network["database"] == "NicheNet_LT")
        & np.array([fr in settings_CV["forbidden_ligands_nichenet"] for fr in gr_network["from"]])
    )
    & ~(
        (gr_network["database"] == "CytoSig")
        & np.array([fr in settings_CV["forbidden_ligands_cytosig"] for fr in gr_network["from"]])
    )
]

In [12]:
source_names = sorted(set(chain(gr_network["source"], lr_network["source"], sig_network["source"])))


def objective(trial: Trial):
    source_weights = {source_name: trial.suggest_float(name=source_name, low=0, high=1) for source_name in source_names}
    lr_sig_hub = trial.suggest_float(name="lr_sig_hub", low=0, high=1)
    gr_hub = trial.suggest_float(name="gr_hub", low=0, high=1)
    ltf_cutoff = trial.suggest_float(name="ltf_cutoff", low=0.9, high=0.999)
    damping_factor = trial.suggest_float(name="damping_factor", low=0.01, high=0.99)
    res = construct_and_evaluate(
        source_weights,
        lr_sig_hub,
        gr_hub,
        ltf_cutoff,
        damping_factor,
        lr_network,
        gr_network,
        sig_network,
        evaluation_data,
    )
    return (res[1], res[2])


study = create_study(sampler=GPSampler(), directions=["maximize", "maximize"])
study.optimize(objective, n_trials=1, n_jobs=1)

[I 2026-10-23 11:34:16,672] A new study created in memory with name: no-name-0efb39f4-5dc2-43d8-a8ba-14f6142d2281
C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\parameter_optimization.py:111: UserWarning: Could not evaluate ligand importance scores for GSE2059_IL15_IL21:
There are no true samples in response. aupr, auroc and pearson correlation coëfficient are undefined.
Traceback (most recent call last):
  File "C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\parameter_optimization.py", line 31, in _evaluate_single_importances_ligand_prediction
    return evaluate_single_importances_ligand_prediction(ligand_importances, group, allow_nan=True)
  File "C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\evaluation.py", line 440, in evaluate_single_importances_ligand_prediction
    zip(
    ~~~^
        *(
        ^^
    ...<13 lines>...
        strict=True,
        ^^^^^^^^^^^^
    ),
    ^
  File "C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\evaluation.py", line 445

the optimization results in a set of pareto optimal solutions

In [13]:
best_params = [trial.params for trial in study.best_trials]
len(best_params)

1

In [14]:
session_info.show()

c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\session_info\main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  mod_version = _find_version(mod.__version__)
